Fine-tune RAVDESS using TESS weights

In [7]:
import os
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from collections import Counter

In [8]:
emotion_map = {
    "01": 0,  
    "02": 1,  
    "03": 2,  
    "04": 3,  
    "05": 4,  
    "06": 5, 
    "07": 6,  
    "08": 7   
}

In [10]:
ravdess_path = r"C:\Users\Thimathi\source\Aurevia\data\raw\RAVDEES"

X = []
y = []

for actor_folder in os.listdir(ravdess_path):
    actor_path = os.path.join(ravdess_path, actor_folder)

    if os.path.isdir(actor_path):
        for file in os.listdir(actor_path):
            if file.endswith(".wav"):
                file_path = os.path.join(actor_path, file)

                emotion_code = file.split("-")[2]
                label = emotion_map[emotion_code]

                audio, sr = librosa.load(file_path, sr=None)

                mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
                mel = librosa.power_to_db(mel, ref=np.max)

                max_len = 128
                if mel.shape[1] < max_len:
                    pad_width = max_len - mel.shape[1]
                    mel = np.pad(mel, ((0,0),(0,pad_width)), mode='constant')
                else:
                    mel = mel[:, :max_len]

                X.append(mel)
                y.append(label)

X = np.array(X)
y = np.array(y)

print("Total samples:", len(X))
print("Class distribution:", Counter(y))

c:\Users\Thimathi\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total samples: 1440
Class distribution: Counter({np.int64(1): 192, np.int64(2): 192, np.int64(3): 192, np.int64(4): 192, np.int64(5): 192, np.int64(6): 192, np.int64(7): 192, np.int64(0): 96})


In [11]:
X = X[:, np.newaxis, :, :]

In [12]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [13]:
mean = np.mean(X_train)
std = np.std(X_train)

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std

In [14]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)

In [15]:
class VoiceCNN(nn.Module):
    def __init__(self, num_classes=8):
        super(VoiceCNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2,2)

        self.fc1 = nn.Linear(128 * 16 * 16, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))

        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

In [ ]:
model = VoiceCNN(num_classes=7)
model.load_state_dict(torch.load("best_tess_model.pth"))

model.fc2 = nn.Linear(256, 8)

In [17]:
for param in model.conv1.parameters():
    param.requires_grad = False
for param in model.conv2.parameters():
    param.requires_grad = False
for param in model.conv3.parameters():
    param.requires_grad = False

In [18]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

In [19]:
num_epochs = 8
batch_size = 32

for epoch in range(num_epochs):
    model.train()

    for i in range(0, len(X_train), batch_size):
        xb = X_train[i:i+batch_size]
        yb = y_train[i:i+batch_size]

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        outputs = model(X_val)
        _, preds = torch.max(outputs, 1)
        val_acc = accuracy_score(y_val.numpy(), preds.numpy())

    print(f"[Stage 1] Epoch {epoch+1}, Val Accuracy: {val_acc:.4f}")

[Stage 1] Epoch 1, Val Accuracy: 0.1944
[Stage 1] Epoch 2, Val Accuracy: 0.2153
[Stage 1] Epoch 3, Val Accuracy: 0.2500
[Stage 1] Epoch 4, Val Accuracy: 0.2778
[Stage 1] Epoch 5, Val Accuracy: 0.2986
[Stage 1] Epoch 6, Val Accuracy: 0.3021
[Stage 1] Epoch 7, Val Accuracy: 0.2778
[Stage 1] Epoch 8, Val Accuracy: 0.3542


In [20]:
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=0.00005)

In [21]:
num_epochs = 15

for epoch in range(num_epochs):
    model.train()

    for i in range(0, len(X_train), batch_size):
        xb = X_train[i:i+batch_size]
        yb = y_train[i:i+batch_size]

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        outputs = model(X_val)
        _, preds = torch.max(outputs, 1)
        val_acc = accuracy_score(y_val.numpy(), preds.numpy())

    print(f"[Stage 2] Epoch {epoch+1}, Val Accuracy: {val_acc:.4f}")

[Stage 2] Epoch 1, Val Accuracy: 0.3472
[Stage 2] Epoch 2, Val Accuracy: 0.3646
[Stage 2] Epoch 3, Val Accuracy: 0.3646
[Stage 2] Epoch 4, Val Accuracy: 0.3507
[Stage 2] Epoch 5, Val Accuracy: 0.3542
[Stage 2] Epoch 6, Val Accuracy: 0.3854
[Stage 2] Epoch 7, Val Accuracy: 0.3924
[Stage 2] Epoch 8, Val Accuracy: 0.3819
[Stage 2] Epoch 9, Val Accuracy: 0.3958
[Stage 2] Epoch 10, Val Accuracy: 0.4132
[Stage 2] Epoch 11, Val Accuracy: 0.3889
[Stage 2] Epoch 12, Val Accuracy: 0.3785
[Stage 2] Epoch 13, Val Accuracy: 0.4097
[Stage 2] Epoch 14, Val Accuracy: 0.4236
[Stage 2] Epoch 15, Val Accuracy: 0.4410
